# Connecting to Estates data lakehouse

This notebooks walks through the installation and connection process for accessing data from a ducklake hosted via UoE [DataSync](https://information-services.ed.ac.uk/computing/desktop-personal/datasync) and an on-premises catalog database, using a Windows PC. It also provides some example queries.

You can check out some docs for the available data [here](http://129.215.207.51:8001/#!/overview).

> Note: you will need to have network access and an access key to the catalog database, and access granted to the storage layer, to use this guide.

The lakehouse implements a [Ducklake](https://ducklake.select/). You can read more about the available features it supports, and how to use them, in the [docs](https://ducklake.select/docs/stable/). Interacting with the lakehouse means writing SQL, which is executed by DuckDB's query engine. If you are unfamiliar with SQL a good starting point is the [DuckDB introduction to SQL](https://duckdb.org/docs/current/sql/introduction). You can read more about the specifics of the DuckDB flavour of SQL [here](https://duckdb.org/docs/current/sql/dialect/overview).
## Installation

It's recommended to access the DataSync storage layer via [rclone](https://rclone.org/), due to limitations in the built-in Windows WebDAV client. This requires [installing rclone](https://rclone.org/install/) and [WinFsp](https://winfsp.dev/rel/).

You can skip this step and go straight to Mounting if you would prefer to use the Windows client.

#### rclone

Install rclone using [their instructions](https://rclone.org/install/), or using winget:

In [ ]:
!winget install Rclone.Rclone

#### WinFsp
Install WinFsp using the [instructions](https://winfsp.dev/rel/), or again using winget:

In [ ]:
!winget install -e --id WinFsp.WinFsp

## Mounting

### Using rclone

If you installed rclone, ensure you have a `.env` file like the following in the current directory with your credentials included. You can generate the value for `RCLONE_WEBDAV_PASS` by pasting your password in the next cell and running it:

In [ ]:
!rclone obscure "<your password here>"

Create a `.env` with the following structure:


Run the following cell to mount your DataSync directory to the `L:` drive, using rclone.

> Note: this cell will continue running for the duration of the mount. If you stop its execution, the drive will unmount. If you would like to run the mount in an external terminal instead, you can open a powershell window and run the `rclone_mount.ps1` file from this directory (or drag the `rclone_mount.ps1` file into the integrated terminal in vscode).

In [ ]:
import dotenv
import os

dotenv.load_dotenv()

print(f"Rclone WebDAV User: {os.getenv("RCLONE_WEBDAV_USER")}")

import subprocess

with subprocess.Popen(
    ['rclone', 'mount', ':webdav:', 'L:', '--vfs-cache-mode', 'writes', '--volname', 'DataSync'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
) as proc:
    for line in proc.stdout: # type: ignore
        print(line, end='', flush=True)

Rclone WebDAV User: estnet0
2026/04/23 13:02:19 NOTICE: Config file "C:\\Users\\ajohns3\\AppData\\Roaming\\rclone\\rclone.conf" not found - using defaults
2026/04/23 13:02:19 ERROR : symlinks not supported without the --links flag: /
The service rclone has been started.


### Using Windows WebDAV

Follow the instructions at https://information-services.ed.ac.uk/computing/desktop-personal/datasync/webdav to mount your DataSync directory as a local file. **Ensure and mount using the `L:` drive letter.**

### Verifying mount

If you navigate to `L:/` in the file explorer, you should see your DataSync files. This should include a `L:\datalake\EST` folder if you have access to the lake.

## Attaching to the lakehouse

### Catalog credentials

Add the following to the `.env` file in the current folder:

### DuckDB connection

Run the cell below to create a duckdb python connection to the lakehouse.

In [2]:
import duckdb
import os
import dotenv

dotenv.load_dotenv()

NETBIRD_HOST = os.getenv("NETBIRD_HOST")
CATALOG_DB_PORT = os.getenv("CATALOG_DB_PORT")
CATALOG_DB_USERNAME = os.getenv("CATALOG_DB_USERNAME")
DATA_PATH = 'L:/datalake/EST/'

print(f"Connecting to ducklake using catalog database at {NETBIRD_HOST}:{CATALOG_DB_PORT} with username {CATALOG_DB_USERNAME}")

con = duckdb.connect()
# note: to create persistent secrets that are available in other duckdb sessions on this machine, use PERSISTENT SECRET instead of SECRET
con.execute(f"""
    CREATE OR REPLACE SECRET ducklake_catalog 
            ( TYPE postgres, HOST '{NETBIRD_HOST}', PORT {CATALOG_DB_PORT}, 
            DBNAME 'Lakehouse_Catalog', USER '{CATALOG_DB_USERNAME}', PASSWORD '{os.getenv("CATALOG_DB_PASSWORD")}' );
""")
con.execute(f"""
 CREATE OR REPLACE SECRET 
        ( TYPE ducklake, DATA_PATH '{DATA_PATH}', METADATA_PATH '', 
            METADATA_PARAMETERS MAP {{'TYPE': 'postgres', 'SECRET': 'ducklake_catalog'}} );
""")
# note: if you change the above commands to use PERSISTENT SECRET, the below ATTACH command will work in any duckdb session on this machine.
con.execute("ATTACH 'ducklake:' AS uoe_est_lakehouse;")
con.execute("USE uoe_est_lakehouse;")

Connecting to ducklake using catalog database at 100.105.144.159:5433 with username read_only_user


> Note: You can attach the lakehouse using a different alias, i.e. not `uoe_est_lakehouse`, but you will not be able to query views from the lakehouse. You will still be able to return data from regular tables.

## Examples

The below cells are example python scripts that demonstrate how to interact with the lakehouse.

In [28]:
schemas_query = con.execute("select distinct schema from (show all tables) where database != '__ducklake_metadata_uoe_est_lakehouse';")
schemas = schemas_query.df()
schemas.head(6)

,schema
0,raw_metabase
1,dbt_marts
2,dbt_int
3,dbt_staging
4,load
5,_dlt_staging_


In [ ]:
# print tables in dbt_marts
tables_query = con.execute("show tables from dbt_marts;")

tables = tables_query.df()
tables.head(6)

,name
0,uoeap__utility_consumption
1,uoeap__utility_consumption__dbt_backup


In [74]:
# describe columns in dbt_marts.uoeap__utility_consumption
tables_query = con.execute("describe dbt_marts.uoeap__utility_consumption;")
tables = tables_query.df()
tables.head(20)

,column_name,column_type,null,key,default,extra
0,reading_datetime,TIMESTAMP WITH TIME ZONE,YES,None,NULL,None
1,reading_date,TIMESTAMP WITH TIME ZONE,YES,None,NULL,None
2,consumption,DOUBLE,YES,None,NULL,None
3,meter_reference,VARCHAR,YES,None,NULL,None
4,meter_channel_id,BIGINT,YES,None,NULL,None
5,channel_description,VARCHAR,YES,None,NULL,None
6,meter_type,VARCHAR,YES,None,NULL,None
7,meter_status,VARCHAR,YES,None,NULL,None
8,reading_status,VARCHAR,YES,None,NULL,None
9,account_units,VARCHAR,YES,None,NULL,None


In [8]:
# get meters in old college

old_college_meters_query = con.execute("""
select distinct meter_reference, channel_description, account_units, is_main_site_utility_account
from dbt_marts.uoeap__utility_consumption
where site_code = '0001';
""")
old_college_meters = old_college_meters_query.df()
old_college_meters.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,meter_reference,channel_description,account_units,is_main_site_utility_account
0,0001NH001M,Heat Meter (kWh) RAW,kWh,False
1,0001NW002M,Water (Cum),Cu Metres,False
2,0001NE001V,Electricity (kWh),kWh,True
3,1800035330472,Electricity (kVArh),kWh,False
4,0001NE004S,Electricity (kWh),kWh,False
5,M1002481,Water (Cum),Cu Metres,True
6,0001NH002M,Heat Meter (kWh) RAW,kWh,False
7,0001NE002V,Electricity (kWh),kWh,False
8,0001NH002M,Heat Meter (kWh),kWh,False
9,0001NH003V,Heat Meter (kWh),kWh,True


In [7]:
# energy by utility in old college for the last 365 days
energy_by_utility_query = con.execute("""
select channel_description, sum(consumption) as total_consumption
from dbt_marts.uoeap__utility_consumption
where site_code = '0001'
and reading_date >= current_date - interval '365' day
and is_main_site_utility_account = true
group by channel_description;
""")
energy_by_utility = energy_by_utility_query.df()
energy_by_utility.head(20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,channel_description,total_consumption
0,Electricity (kWh),4.745007e+05
1,Heat Meter (kWh),1.301780e+06
2,Water (Cum),4.701381e+03


In [ ]:
# save all meter readings for old college to a dataframe

old_college_readings_query = con.execute("""
from dbt_marts.uoeap__utility_consumption
where site_code = '0001'
order by reading_date asc
""")
old_college_readings = old_college_readings_query.df()
old_college_readings.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,reading_datetime,reading_date,consumption,meter_reference,meter_channel_id,channel_description,meter_type,meter_status,reading_status,account_units,site_code,site_name,is_main_site_utility_account,row_version
0,2002-04-01 01:30:00+01:00,2002-04-01 00:00:00+01:00,5.0,1800035330463,4824,Electricity (kVArh),Main,ACTIVE,N,kWh,0001,0001 OLD COLLEGE,False,76593353
1,2002-04-01 02:00:00+01:00,2002-04-01 00:00:00+01:00,4.0,1800035330463,4824,Electricity (kVArh),Main,ACTIVE,N,kWh,0001,0001 OLD COLLEGE,False,76593353
2,2002-04-01 02:30:00+01:00,2002-04-01 00:00:00+01:00,4.0,1800035330463,4824,Electricity (kVArh),Main,ACTIVE,N,kWh,0001,0001 OLD COLLEGE,False,76593353
3,2002-04-01 03:00:00+01:00,2002-04-01 00:00:00+01:00,5.0,1800035330463,4824,Electricity (kVArh),Main,ACTIVE,N,kWh,0001,0001 OLD COLLEGE,False,76593353
4,2002-04-01 03:30:00+01:00,2002-04-01 00:00:00+01:00,4.0,1800035330463,4824,Electricity (kVArh),Main,ACTIVE,N,kWh,0001,0001 OLD COLLEGE,False,76593353
